# Influence of the replacement rate $k_r$

Sensitivity of the simulation to the replacement rate of the mutant cell. $k_r$
is doubled and halved around its fitted value of 0.0625 while all remaining
parameters are held fixed, for the conditions BRAF; MHCII fl/fl and
BRAF; MHCII fl/+.

The notebook is part of the `scb2d_analysis` package and uses its modules, as
does `plot_simulation_results_publication.ipynb`, which contains the main
figures. The first cell adds the directory containing the package to
`sys.path`, so that the imports resolve independently of the directory the
kernel is started in.

## Applying the analysis to another parameter

The analysis is not specific to $k_r$. The runs are described by a single nested
dictionary

```python
{condition: {variant: run folder}}
```

constructed with `datasets.build_sensitivity_runs`. To examine a different
parameter, assign a dictionary of the same shape to `SENSITIVITY_RUNS` in the
following cell; the figures, colors and legends are derived from it. Each
condition is assigned a shade family: the `reference` variant is drawn in the
base shade with a $\pm$1 SD band, the remaining variants in the lighter and the
darker shade of that family.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# The notebook is located inside the package, therefore the directory
# *containing* `scb2d_analysis` has to be on the import path. It is located by
# walking up from the working directory.
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "scb2d_analysis" / "__init__.py").exists():
        if str(folder) not in sys.path:
            sys.path.insert(0, str(folder))
        break

from scb2d_analysis import datasets, experimental_data, fixation, io_simulation
from scb2d_analysis import monoclonality, plots
from scb2d_analysis.plot_style import (
    CM, LINE_STYLES, P_MONOCLONAL_GIVEN_VISIBLE_LABEL, save_figure,
    setup_plot_style, shade_family,
)

plt.rcdefaults()

# Font of the panels. matplotlib reads these settings when a figure is created,
# so they are applied here rather than in the figure cells; a `setup_plot_style`
# call after `plt.subplots` would not reach the ticks of that figure.
plt.rcParams.update({"font.size": 9, "font.family": "Arial"})

# Export of the panels. Set to True to write every figure into FIGURE_DIR;
# `run_notebooks.py` turns it on through the environment.
SAVE_FIGURES = os.environ.get("SCB2D_SAVE_FIGURES") == "1"
FIGURE_DIR = Path(datasets.__file__).parent / "figures"

# --- Configuration of the analysis -------------------------------------------
# The runs to be analysed, as {condition: {variant: run folder}}; see
# `datasets.build_sensitivity_runs` for how to define a different set. The
# variant named `datasets.REFERENCE_VARIANT` ("reference") is the run the
# remaining variants are compared against.
SENSITIVITY_RUNS = datasets.KR_SENSITIVITY

# Legend entry of each variant. Variants without an entry are labeled with their
# own key.
VARIANT_LABELS = {
    "kr x 2": r"$k_r$ * 2",
    "kr / 2": r"$k_r$ / 2",
}

# Report configured but unreachable folders before any analysis is run.
missing = datasets.sensitivity_missing_paths(SENSITIVITY_RUNS)
if missing:
    raise FileNotFoundError("configured but missing:\n  " + "\n  ".join(missing))

for condition, run_folders in SENSITIVITY_RUNS.items():
    print(f"{condition}: {', '.join(run_folders)}")

## Curves to be drawn

One entry per (condition, variant), holding what the figures below require: the
run folder, the color, the line style and the legend entry. The table is derived
from `SENSITIVITY_RUNS`, so that a different set of runs requires no change
here.

In [ ]:
# Line style of each variant, in the order in which the variants are listed for
# a condition: solid for the reference run, dashed and dotted for the variants.
VARIANT_LINE_STYLES = [LINE_STYLES[0], LINE_STYLES[1], LINE_STYLES[3]]

curves = []

for condition_index, (condition, run_folders) in enumerate(SENSITIVITY_RUNS.items()):
    shades = shade_family(condition_index)

    for variant_index, (variant, run_folder) in enumerate(run_folders.items()):
        is_reference = variant == datasets.REFERENCE_VARIANT

        curves.append({
            "key": (condition, variant),
            "condition": condition,
            "variant": variant,
            "is_reference": is_reference,
            "run_folder": run_folder,
            "color": shades[variant_index % len(shades)],
            "line_style": VARIANT_LINE_STYLES[variant_index % len(VARIANT_LINE_STYLES)],
            # The reference run is labeled with the condition, the variants with
            # the parameter change.
            "label": condition if is_reference else VARIANT_LABELS.get(variant, variant),
        })

for curve in curves:
    print(f"{curve['label']:<20} {curve['color']}  {curve['line_style']}")

## Fixation and extinction over time

`fixation_events[(condition, variant)]` holds, for every crypt of every
simulation run, the day it became monoclonal and the founder lineage that took
it over. Reading the folders takes a while, so it is done once here and reused
by both figures below.

In [ ]:
fixation_events = {}

for curve in curves:
    fixation_events[curve["key"]] = io_simulation.read_fixation_events_from_directory(
        datasets.pos_and_time_folder(curve["run_folder"])
    )
    print(f"{curve['key']}: {len(fixation_events[curve['key']])} simulation runs")

In [ ]:
def probability_curve(grouping_function, key, max_day):
    """
    Cumulative probability of one set of runs over the simulated days.

    Parameters
    ----------
    grouping_function : callable
        `fixation.group_fixation_events_by_day` (a labeled lineage takes over a
        crypt) or `fixation.group_extinction_events_by_day` (a labeled
        lineage of a crypt dies out).
    key : tuple
        (condition, variant), identifying the runs in `fixation_events`.
    max_day : int
        Last day to be evaluated. Its mean, standard deviation and confidence
        interval are printed.

    Returns
    -------
    days, means, stds : list
        Output of `fixation.average_probability_curves`.
    """
    events_by_day = grouping_function(fixation_events[key], max_day=max_day)
    cumulative_per_run = fixation.compute_cumulative_probability_per_run(events_by_day)

    days, means, stds = fixation.average_probability_curves(cumulative_per_run)
    *_, confidence_intervals = fixation.average_probability_curves_with_ci(cumulative_per_run)

    condition, variant = key
    print(f"{condition} / {variant} - "
          f"{fixation.format_probability_at_day(days, means, stds, confidence_intervals, max_day)}")

    return days, means, stds


def annotate_curve_end(annotations, curve, means):
    """
    Write the label of a curve next to its last value.

    Parameters
    ----------
    annotations : dict
        (condition, variant) -> (x position, offset from the last value of the
        curve). Curves without an entry are identified by the legend alone.
    curve : dict
        One entry of `curves`.
    means : sequence
        Mean probability per day, as returned by `probability_curve`.
    """
    if curve["key"] not in annotations:
        return

    x_position, vertical_offset = annotations[curve["key"]]
    plt.text(x_position, means[-1] + vertical_offset, curve["label"],
             fontsize=9, color=curve["color"])


# The ±1 SD band is drawn for the reference run only; the variants are drawn as
# plain lines.
REFERENCE_BAND_ALPHA = 0.2

LAST_PLOTTED_DAY = 60

### Figure: fixation probability

Cumulative probability that a crypt has been taken over by a *labeled* lineage,
averaged over simulation runs. The band is $\pm$1 SD of the reference run.

In [ ]:
# Position of the label drawn next to the end of a curve:
# (condition, variant) -> (x position, offset from the last value of the curve).
# The annotations are optional; a curve without an entry is identified by the
# legend alone.
FIXATION_ANNOTATIONS = {
    ("BRAF; MHCII fl/fl", "kr x 2"): (50, -0.07),
    ("BRAF; MHCII fl/fl", "kr / 2"): (50, 0.05),
    ("BRAF; MHCII fl/+", "kr x 2"): (50, -0.06),
    ("BRAF; MHCII fl/+", "kr / 2"): (50, 0.04),
}

plt.subplots(figsize=(8.7 * CM, 6.66 * CM), constrained_layout=True)
setup_plot_style(font_size=9)

for curve in curves:
    days, means, stds = probability_curve(
        fixation.group_fixation_events_by_day, curve["key"], LAST_PLOTTED_DAY
    )

    plots.plot_mean_with_std_band(
        days, means, stds,
        band_color=curve["color"], line_color=curve["color"],
        label=curve["label"], line_style=curve["line_style"],
        band_alpha=REFERENCE_BAND_ALPHA if curve["is_reference"] else 0,
    )
    annotate_curve_end(FIXATION_ANNOTATIONS, curve, means)

plt.ylim(0, 1)
plt.xlim(0, LAST_PLOTTED_DAY)
plt.ylabel("Fixation Probability")
plt.xlabel("Days")
plt.legend(loc="center right", bbox_to_anchor=(1, 0.21), frameon=False,
           fontsize=8, labelspacing=0.2)

if SAVE_FIGURES:
    save_figure("kr_fixation_probability", FIGURE_DIR)

### Figure: extinction probability

Cumulative probability that all labeled lineages of a crypt died out, i.e. the
crypt was taken over by an unlabeled cell.

In [ ]:
EXTINCTION_ANNOTATIONS = {
    ("BRAF; MHCII fl/fl", "kr x 2"): (54, 0.03),
    ("BRAF; MHCII fl/fl", "kr / 2"): (54, -0.09),
    ("BRAF; MHCII fl/+", "kr x 2"): (45, 0.02),
    ("BRAF; MHCII fl/+", "kr / 2"): (45, -0.08),
}

plt.subplots(figsize=(8.7 * CM, 6.66 * CM), constrained_layout=True)
setup_plot_style(font_size=9)

for curve in curves:
    days, means, stds = probability_curve(
        fixation.group_extinction_events_by_day, curve["key"], LAST_PLOTTED_DAY
    )

    plots.plot_mean_with_std_band(
        days, means, stds,
        band_color=curve["color"], line_color=curve["color"],
        label=curve["label"], line_style=curve["line_style"],
        band_alpha=REFERENCE_BAND_ALPHA if curve["is_reference"] else 0,
    )
    annotate_curve_end(EXTINCTION_ANNOTATIONS, curve, means)

plt.ylim(0, 0.9)
plt.xlim(0, LAST_PLOTTED_DAY)
plt.ylabel("Extinction Probability")
plt.xlabel("Days")
plt.legend(loc="upper left", bbox_to_anchor=(-0.01, 1.06), frameon=False,
           fontsize=8, labelspacing=0.2)

if SAVE_FIGURES:
    save_figure("kr_extinction_probability", FIGURE_DIR)

## Figure: P(Monoclonal | Visible) over time

Simulated curves against the measured data points of the reference condition.
The simulated values are read from the `percent_fixed` subfolder of the same
runs.

`MONOCLONALITY_CONDITIONS` determines which conditions are included in this
panel; set it to `list(SENSITIVITY_RUNS)` to include all of them.

In [ ]:
measured = experimental_data.load_experimental_data_if_available(
    datasets.EXPERIMENTAL_DATA_FILE
)

# Conditions included in this panel, in drawing order.
MONOCLONALITY_CONDITIONS = ["BRAF; MHCII fl/fl"]

# Condition -> key of the measured data in the Excel sheet. Conditions without
# an entry, and all conditions if the experimental data are unavailable, are
# drawn without data points.
MEASURED_KEYS = {
    "BRAF; MHCII fl/fl": "block2_BRAF; MHCII fl/fl",
    "BRAF; MHCII fl/+": "block2_BRAF; MHCII fl/+",
}

# Legend entries of this panel, stating the parameter change explicitly.
MONOCLONALITY_VARIANT_LABELS = {
    datasets.REFERENCE_VARIANT: r"$k_r^{\mu}=k_r^{wt}$",
    "kr x 2": r"$k_r^{\mu}=k_r^{wt}*2$",
    "kr / 2": r"$k_r^{\mu}=k_r^{wt}/2$",
}

# Appended to the legend entry of the reference run of a condition.
CONDITION_NOTES = {
    "BRAF; MHCII fl/fl": r"$k_d^{\mu}\sim k_d^{wt}*3.3$",
}

if measured:
    print(f"{len(measured)} experimental conditions loaded")
else:
    print("no experimental data available; the figure is drawn without the "
          "measured data points")

In [ ]:
plt.subplots(figsize=(7.65 * CM, 5.83 * CM), constrained_layout=True)
setup_plot_style(font_size=9)

for condition in MONOCLONALITY_CONDITIONS:
    for curve in [curve for curve in curves if curve["condition"] == condition]:
        probabilities_by_day = io_simulation.read_visible_monoclonal_probabilities_from_directory(
            datasets.percent_fixed_folder(curve["run_folder"])
        )
        mean_by_day, std_by_day = monoclonality.average_probability_per_day(probabilities_by_day)

        label = MONOCLONALITY_VARIANT_LABELS.get(curve["variant"], curve["label"])

        if curve["is_reference"]:
            # Measured data points, plus a marker entry naming the condition.
            block = measured.get(MEASURED_KEYS.get(condition, ""))
            if block:
                plots.plot_experimental_points(block["Days"], block["Avg"], block["SD"],
                                               curve["color"])
                plt.scatter([], [], marker="o", color=curve["color"], label=condition)

            note = CONDITION_NOTES.get(condition)
            if note:
                label = f"{label}, {note}"

        plots.plot_probability_over_days(
            mean_by_day, std_by_day, curve["line_style"], curve["color"], label,
            # Only the reference run gets a band, as in the panels above.
            band_alpha=0.3 if curve["is_reference"] else 0,
        )

legend = plt.legend(loc="upper left", bbox_to_anchor=(0, 1.08), fontsize=9,
                    frameon=False, labelspacing=0.15, handlelength=1.2,
                    handletextpad=0.3, borderaxespad=0.1)
legend.set_in_layout(False)

plt.xlim(0, 25)
plt.ylim(0, 1)
plt.ylabel(P_MONOCLONAL_GIVEN_VISIBLE_LABEL)
plt.xlabel("Days")

if SAVE_FIGURES:
    save_figure("kr_p_monoclonal_visible", FIGURE_DIR)